# HydraShield Fire Risk Analysis

This notebook demonstrates the HydraShield prediction pipeline: estimating fuel moisture content, computing fire spread, and assessing wildfire risk.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

from src.prediction.fuel_moisture import FuelMoistureModel
from src.prediction.fire_spread import FireSpreadModel
from src.prediction.risk_model import WildfireRiskModel

## 1. Fuel Moisture Content Estimation

Estimate FMC from NDMI and model capillary transfer from soil moisture.

In [ ]:
fmc_model = FuelMoistureModel()

# Simulated NDMI values across a landscape
ndmi = np.linspace(-0.5, 0.5, 100)
fmc_ndmi = fmc_model.estimate_fmc_from_ndmi(ndmi)

plt.figure(figsize=(8, 4))
plt.plot(ndmi, fmc_ndmi)
plt.xlabel('NDMI')
plt.ylabel('Estimated FMC (%)')
plt.title('FMC vs NDMI')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Fire Spread Modelling

Compute Rate of Spread as a function of FMC, wind, and slope.

In [ ]:
spread_model = FireSpreadModel(fuel_model='TL3')

fmc_range = np.linspace(5, 25, 50)
ros_reduced = [spread_model.compute_ros(f, 20.0, 10.0).ros_reduced for f in fmc_range]

plt.figure(figsize=(8, 4))
plt.plot(fmc_range, ros_reduced)
plt.xlabel('FMC (%)')
plt.ylabel('Rate of Spread (m/min)')
plt.title('ROS vs FMC (wind=20 km/h, slope=10 deg)')
plt.grid(True, alpha=0.3)
plt.show()

## 3. ML Risk Model

Train a Random Forest on synthetic historical fire data.

In [ ]:
rng = np.random.RandomState(42)
X = rng.rand(500, 4)
y = (X[:, 0] + X[:, 1] > 1.0).astype(int)

risk_model = WildfireRiskModel(n_estimators=50, random_state=42)
metrics = risk_model.train(
    X, y,
    feature_names=['temperature', 'wind', 'humidity', 'fuel_load']
)

print('Validation metrics:')
for k, v in metrics.to_dict().items():
    print(f'  {k}: {v:.3f}')

print('\nFeature importances:')
for k, v in risk_model.feature_importances().items():
    print(f'  {k}: {v:.3f}')